# Week 9 — RFM Customer Segmentation
**Phase 3: Business Metrics · ~15 minutes · Tables: `OLIST_ORDERS`, `OLIST_ORDER_PAYMENTS`**

---

## What you're practicing this week

RFM is a classic customer segmentation framework. It scores every customer on three dimensions and groups them into segments. It's a great capstone for Phase 3 — it uses CTEs, window functions, aggregations, and CASE WHEN all together.

## The RFM framework

| Dimension | Measures | How we calculate it |
|---|---|---|
| **R**ecency | How recently did they last order? | Days since their last order |
| **F**requency | How often do they order? | Total number of orders |
| **M**onetary | How much do they spend? | Total payment value |

### Scoring with NTILE
`NTILE(5)` divides rows into 5 equal buckets — score 5 is best, score 1 is worst:

```sql
-- Score frequency: more orders = higher score
NTILE(5) OVER (ORDER BY frequency ASC) AS f_score

-- Score recency: fewer days since last order = higher score
-- ORDER BY DESC so the most recent customers get score 5
NTILE(5) OVER (ORDER BY recency_days DESC) AS r_score
```

### Segments
```sql
CASE
    WHEN r_score >= 4 AND f_score >= 4 AND m_score >= 4 THEN 'Champion'
    WHEN f_score >= 4 AND r_score >= 3               THEN 'Loyal'
    WHEN r_score <= 2 AND f_score >= 3               THEN 'At Risk'
    WHEN r_score >= 4 AND f_score = 1                THEN 'New Customer'
    WHEN r_score = 1  AND f_score <= 2               THEN 'Lost'
    ELSE 'Other'
END AS segment
```

---

## Try the examples

In [ ]:
-- Example 1: calculate raw RFM values
-- Reference date = the latest order date in the dataset
WITH ref_date AS (
    SELECT MAX(DATE_TRUNC('day', order_purchase_timestamp)) AS max_date
    FROM ANALYTICS_TEAM_DB.SQL_CHALLENGE.OLIST_ORDERS
),
raw_rfm AS (
    SELECT
        o.customer_id,
        DATEDIFF('day', MAX(DATE_TRUNC('day', o.order_purchase_timestamp)), r.max_date) AS recency_days,
        COUNT(DISTINCT o.order_id) AS frequency,
        SUM(p.payment_value)       AS monetary
    FROM ANALYTICS_TEAM_DB.SQL_CHALLENGE.OLIST_ORDERS o
    JOIN ANALYTICS_TEAM_DB.SQL_CHALLENGE.OLIST_ORDER_PAYMENTS p ON o.order_id = p.order_id
    CROSS JOIN ref_date r
    GROUP BY o.customer_id, r.max_date
)
SELECT * FROM raw_rfm LIMIT 10;

In [ ]:
-- Example 2: score each dimension with NTILE
WITH ref_date AS (
    SELECT MAX(DATE_TRUNC('day', order_purchase_timestamp)) AS max_date
    FROM ANALYTICS_TEAM_DB.SQL_CHALLENGE.OLIST_ORDERS
),
raw_rfm AS (
    SELECT o.customer_id,
        DATEDIFF('day', MAX(DATE_TRUNC('day', o.order_purchase_timestamp)), r.max_date) AS recency_days,
        COUNT(DISTINCT o.order_id) AS frequency,
        SUM(p.payment_value) AS monetary
    FROM ANALYTICS_TEAM_DB.SQL_CHALLENGE.OLIST_ORDERS o
    JOIN ANALYTICS_TEAM_DB.SQL_CHALLENGE.OLIST_ORDER_PAYMENTS p ON o.order_id = p.order_id
    CROSS JOIN ref_date r
    GROUP BY o.customer_id, r.max_date
)
SELECT
    customer_id,
    NTILE(5) OVER (ORDER BY recency_days DESC) AS r_score,
    NTILE(5) OVER (ORDER BY frequency ASC)    AS f_score,
    NTILE(5) OVER (ORDER BY monetary ASC)     AS m_score
FROM raw_rfm
LIMIT 10;

---

## Task 1 — Calculate raw RFM values per customer

For each customer calculate: recency (days since last order vs dataset max date), frequency (distinct orders), and monetary (total payments). Return `customer_id`, `recency_days`, `frequency`, `monetary`. Use CTEs.

> **What to think about:** For recency, lower is better. For frequency and monetary, higher is better. This matters when you score them.

In [ ]:
-- Task 1: your query here


In [ ]:
-- Submit Task 1
CALL ANALYTICS_TEAM_DB.SQL_CHALLENGE.SUBMIT(
    CURRENT_USER(),
    9,
    1,
    $$
        -- paste your Task 1 query here
    $$
);

---

## Task 2 — Score each dimension 1–5

Using `NTILE(5)`, score each RFM dimension. For recency, reverse the score so fewer days = higher score. Return `customer_id`, `r_score`, `f_score`, `m_score`.

> **What to think about:** Build this as a CTE on top of Task 1 so your logic stays clean and readable.

In [ ]:
-- Task 2: your query here


In [ ]:
-- Submit Task 2
CALL ANALYTICS_TEAM_DB.SQL_CHALLENGE.SUBMIT(
    CURRENT_USER(),
    9,
    2,
    $$
        -- paste your Task 2 query here
    $$
);

---

## Task 3 — Assign customer segments

Using the scores from Task 2, assign each customer a segment using `CASE WHEN`:

- **Champion:** r_score >= 4 AND f_score >= 4 AND m_score >= 4
- **Loyal:** f_score >= 4 AND r_score >= 3
- **At Risk:** r_score <= 2 AND f_score >= 3
- **New Customer:** r_score >= 4 AND f_score = 1
- **Lost:** r_score = 1 AND f_score <= 2
- **Other:** everything else

> **What to think about:** The order of your CASE WHEN conditions matters — a customer can match multiple rules. Which segment should take priority?

In [ ]:
-- Task 3: your query here


In [ ]:
-- Submit Task 3
CALL ANALYTICS_TEAM_DB.SQL_CHALLENGE.SUBMIT(
    CURRENT_USER(),
    9,
    3,
    $$
        -- paste your Task 3 query here
    $$
);

---

## Task 4 — Segment summary

Aggregate your segmented customers: return `segment`, `customer_count`, `avg_monetary`, and `avg_recency_days`. Sort by `avg_monetary` descending.

> **What to think about:** How different is the monetary value between Champions and Loyal customers? Is the At Risk segment large?

In [ ]:
-- Task 4: your query here


In [ ]:
-- Submit Task 4
CALL ANALYTICS_TEAM_DB.SQL_CHALLENGE.SUBMIT(
    CURRENT_USER(),
    9,
    4,
    $$
        -- paste your Task 4 query here
    $$
);

---

## Bonus challenge

Add an `rfm_score` column (r_score + f_score + m_score) and find the top 20 customers by combined RFM score. Include their segment label.

In [ ]:
-- Bonus: your query here


---

*Next week: Data quality — handling NULLs, duplicates, and messy data.*